# Korean sLLM 학습 (Gemma-계열 + 16-토큰 MTP)

GitHub 리포를 clone 하고 tar.xz 데이터를 풀어 학습한다. GPU 런타임(T4 이상)을 선택할 것.

In [ ]:
# 1) 리포 clone (본인 리포 URL 로 수정)
REPO_URL = "https://github.com/<YOUR_ID>/korean_sllm.git"
!git clone {REPO_URL} korean_sllm
%cd korean_sllm
!pip install -q -r requirements.txt

In [ ]:
# 2) (선택) Google Drive 마운트 - 체크포인트 보존용
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/korean_sllm_ckpt'

In [ ]:
# 3) 데이터 압축 해제 (train.py 가 자동으로 풀지만 미리 풀어 확인)
!tar -xJf train.tar.xz && tar -xJf val.tar.xz
!wc -l train.jsonl val.jsonl

In [ ]:
# 4) 학습 (T4 16GB 기준. A100 이면 --batch-size 16 --grad-accum 2 권장)
!python train.py \
  --batch-size 8 --grad-accum 4 \
  --max-steps 20000 --warmup-steps 500 --lr 3e-4 \
  --eval-every 500 --save-every 1000 \
  --grad-checkpointing \
  --ckpt-dir {CKPT_DIR}

# 재개: --resume {CKPT_DIR}/step0XXXXX.pt 추가

In [ ]:
# 5) 생성 데모
import torch
from data import load_tokenizer, encode_sample
from model import KoreanSLLM, ModelConfig

ckpt = torch.load(f'{CKPT_DIR}/final.pt', map_location='cuda', weights_only=True)
model = KoreanSLLM(ModelConfig(**ckpt['config'])).cuda()
model.load_state_dict(ckpt['model'])
sp = load_tokenizer()

prompt = '감기에 걸렸을 때 어떻게 해야 하나요?'
ids = encode_sample(sp, prompt, '')[0][:-2]
out = model.generate(torch.tensor([ids], device='cuda'), max_new_tokens=256, temperature=0.7)
print(sp.decode(out[0].tolist()))